# 6. Symptom Normalization

This notebook normalizes the raw `Symptom` column into canonical symptom patterns using HDBSCAN clustering + LLM labeling.

**Note:** This is different from notebook 1 which clusters `Subjects + Symptom + Caused of Problem` combined. Here we cluster `Symptom` alone to create clean, distinct SymptomPattern nodes for the knowledge graph.

**Output:**
- `output/symptom_patterns.json` — list of canonical symptom patterns
- `output/symptom_mapping.csv` — raw symptom text → canonical pattern
- Updated `output/clustered_emr.csv` with `symptom_cluster_id` and `symptom_canonical` columns

In [1]:
import os
import sys
import json
import re
import numpy as np
import pandas as pd

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.config import settings
from src.utils import get_embeddings, get_llm

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("1. Loading Clustered Data...")
file_path = os.path.join(path_prefix, "output", "clustered_emr.csv")
if not os.path.exists(file_path):
    raise FileNotFoundError("clustered_emr.csv not found. Run notebooks 1 and 5 first.")

df = pd.read_csv(file_path)
print(f"Loaded {len(df)} records.")

# Extract symptom column
symptom_col = "Symptom"
if symptom_col not in df.columns:
    raise ValueError(f"Column '{symptom_col}' not found. Available: {df.columns.tolist()}")

symptoms = df[symptom_col].fillna("").astype(str).str.strip()
valid_mask = symptoms.str.len() > 2
print(f"Valid symptoms: {valid_mask.sum()} / {len(symptoms)}")

1. Loading Clustered Data...
Loaded 20630 records.
Valid symptoms: 20194 / 20630


In [3]:
print("2. Embedding Symptom Texts...")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(settings.embedding_model)
symptom_texts = symptoms[valid_mask].tolist()

embeddings = model.encode(
    symptom_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

2. Embedding Symptom Texts...


Batches: 100%|██████████| 316/316 [01:21<00:00,  3.88it/s]


Embeddings shape: (20194, 384)


In [4]:
print("3. UMAP Dimensionality Reduction...")
import umap

reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)
reduced = reducer.fit_transform(embeddings)
print(f"Reduced shape: {reduced.shape}")

3. UMAP Dimensionality Reduction...


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Reduced shape: (20194, 10)


In [5]:
print("4. HDBSCAN Clustering on Symptoms...")
import hdbscan

# Note: Increase min_cluster_size (e.g., 30 or 50) if you want fewer clusters and much faster LLM labeling
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=settings.hdbscan_min_cluster_size,
    min_samples=settings.hdbscan_min_samples,
    metric="euclidean",
    cluster_selection_method="eom",
)
symptom_labels = clusterer.fit_predict(reduced)

n_clusters = len(set(symptom_labels)) - (1 if -1 in symptom_labels else 0)
n_noise = int((symptom_labels == -1).sum())
print(f"Found {n_clusters} symptom clusters, {n_noise} noise ({n_noise/len(symptom_labels)*100:.1f}%)")

4. HDBSCAN Clustering on Symptoms...
Found 1315 symptom clusters, 3765 noise (18.6%)


In [ ]:
print("5. LLM-Assisted Labeling of Symptom Clusters...")
from langchain_core.messages import HumanMessage

llm = get_llm(temperature=0.0)

symptom_cluster_labels = {}
unique_clusters = sorted(set(symptom_labels))
unique_clusters = [c for c in unique_clusters if c != -1]

for i, cid in enumerate(unique_clusters):
    mask = symptom_labels == cid
    cluster_texts = [symptom_texts[j] for j in range(len(symptom_texts)) if mask[j]]
    
    # Get representative samples
    c_emb = embeddings[mask]
    centroid = c_emb.mean(axis=0)
    distances = np.linalg.norm(c_emb - centroid, axis=1)
    n_sel = min(5, len(cluster_texts))
    closest_idx = np.argsort(distances)[:n_sel]
    samples = [cluster_texts[idx] for idx in closest_idx]
    
    samples_text = "\n".join(f"  {j+1}. {t[:200]}" for j, t in enumerate(samples))
    
    prompt = f"""Kamu adalah expert maintenance alat berat senior.
Berikut {len(samples)} contoh GEJALA/SYMPTOM yang serupa (dikelompokkan oleh algoritma clustering):

{samples_text}

Tugasmu:
1. Analisis pola umum gejala dari contoh-contoh di atas.
2. Berikan SATU label gejala berupa FRASE gejala spesifik yang menyebutkan nama komponen dan jenis indikasi kerusakannya secara detail (maksimal 3-5 kata, bahasa Inggris).
   Contoh gaya penulisan label yang benar: 'Oil Leakage on Final Drive RH', 'Abnormal Noise from Engine', 'Engine Overheating Warning', 'Brake Linkage Loose'.
   JANGAN hanya menuliskan kata generik seperti 'Oil Leak', 'Noise', 'Overheating', atau 'Damage'. Jelaskan letak/komponen spesifiknya.
3. JANGAN meniru contoh gaya penulisan di atas jika tidak sesuai dengan isi teks gejala nyata di atas.

PENTING: Jawab HANYA dalam format JSON:
{{"label": "Symptom Name", "confidence": 0.85}}"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        raw = response.content.strip()
        
        if raw.startswith("```"):
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            
        parsed = json.loads(raw)
        symptom_cluster_labels[cid] = parsed
        
        if (i + 1) % 50 == 0 or (i + 1) == len(unique_clusters):
            print(f"  Progress: {i+1}/{len(unique_clusters)} clusters labeled")
    except Exception as e:
        symptom_cluster_labels[cid] = {"label": f"Symptom_Cluster_{cid}", "confidence": 0.0}
        if (i + 1) % 50 == 0:
            print(f"  Progress: {i+1}/{len(unique_clusters)} (fallback used for {cid})")


5. LLM-Assisted Labeling of Symptom Clusters...


In [7]:
print("6. Building Outputs...")
output_dir = os.path.join(path_prefix, "output")

# --- symptom_patterns.json ---
symptom_patterns = []
for cid in sorted(symptom_cluster_labels.keys()):
    mask = symptom_labels == cid
    info = symptom_cluster_labels[cid]
    symptom_patterns.append({
        "cluster_id": int(cid),
        "label": info["label"],
        "confidence": info.get("confidence", 0),
        "size": int(mask.sum()),
    })

symptom_patterns = sorted(symptom_patterns, key=lambda x: x["size"], reverse=True)
with open(os.path.join(output_dir, "symptom_patterns.json"), "w", encoding="utf-8") as f:
    json.dump(symptom_patterns, f, ensure_ascii=False, indent=2)
print(f"Saved {len(symptom_patterns)} symptom patterns to symptom_patterns.json")

# --- symptom_mapping.csv ---
mapping_data = []
for j, text in enumerate(symptom_texts):
    cid = symptom_labels[j]
    canonical = symptom_cluster_labels.get(cid, {}).get("label", "Uncategorized") if cid != -1 else "Uncategorized"
    mapping_data.append({"raw_symptom": text, "symptom_cluster_id": int(cid), "symptom_canonical": canonical})

mapping_df = pd.DataFrame(mapping_data)
mapping_df.to_csv(os.path.join(output_dir, "symptom_mapping.csv"), index=False, encoding="utf-8-sig")
print(f"Saved symptom_mapping.csv with {len(mapping_df)} rows")

# --- Update clustered_emr.csv ---
df["symptom_cluster_id"] = -1
df["symptom_canonical"] = "Uncategorized"

valid_indices = df.index[valid_mask]
for idx_pos, orig_idx in enumerate(valid_indices):
    cid = int(symptom_labels[idx_pos])
    df.at[orig_idx, "symptom_cluster_id"] = cid
    if cid != -1 and cid in symptom_cluster_labels:
        df.at[orig_idx, "symptom_canonical"] = symptom_cluster_labels[cid]["label"]

df.to_csv(file_path, index=False, encoding="utf-8-sig")
print(f"Updated clustered_emr.csv with symptom_cluster_id and symptom_canonical columns")

6. Building Outputs...
Saved 0 symptom patterns to symptom_patterns.json
Saved symptom_mapping.csv with 20194 rows
Updated clustered_emr.csv with symptom_cluster_id and symptom_canonical columns


In [8]:
print("7. Summary...")
print(f"\nTop 10 Symptom Patterns by frequency:")
for p in symptom_patterns[:10]:
    print(f"  - {p['label']} ({p['size']} records, confidence: {p['confidence']})")

print(f"\nTotal symptom clusters: {len(symptom_patterns)}")
print(f"Uncategorized (noise): {n_noise} records")

7. Summary...

Top 10 Symptom Patterns by frequency:

Total symptom clusters: 0
Uncategorized (noise): 3765 records


In [ ]:
print("8. Recovering Uncategorized Data via Contextual Labeling...")
from langchain_core.messages import HumanMessage
import numpy as np

valid_cids = [c for c in symptom_cluster_labels.keys() if c != -1]
centroids = {}
for cid in valid_cids:
    centroids[cid] = embeddings[symptom_labels == cid].mean(axis=0)

noise_indices = np.where(symptom_labels == -1)[0]
auto_count = 0
llm_count = 0

for i, idx in enumerate(noise_indices):
    raw_text = symptom_texts[idx]
    n_emb = embeddings[idx]
    
    sims = [(cid, np.dot(n_emb, cent)/(np.linalg.norm(n_emb)*np.linalg.norm(cent))) for cid, cent in centroids.items()]
    sims.sort(key=lambda x: x[1], reverse=True)
    if not sims: continue
    
    top_5 = sims[:5]
    best_cid, best_sim = top_5[0]
    matched_cid = -1
    
    if best_sim >= 0.82:
        matched_cid = best_cid
        auto_count += 1
    else:
        options_text = "\n".join([f"{j+1}. {symptom_cluster_labels[cid]['label']}" for j, (cid, sim) in enumerate(top_5)])
        prompt = f"""Anda adalah pakar maintenance alat berat.
Teks asli laporan mekanik: "{raw_text}"

Teks ini sebelumnya tidak terklasifikasi. Berdasarkan konteks gejala kerusakan, pilih SATU label yang paling relevan dan akurat dari 5 daftar kategori standar berikut:
{options_text}

PENTING: Jika teks asli tersebut murni karakter acak (typo parah), tidak masuk akal, atau sama sekali tidak berhubungan dengan 5 opsi di atas, Anda WAJIB menjawab: "Uncategorized".
Output HANYA teks label pilihan Anda tanpa tambahan apapun."""
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            ans = response.content.strip().replace('"', '').replace("'", "")
            
            valid_options = [symptom_cluster_labels[cid]['label'] for cid, sim in top_5]
            matched_label = "Uncategorized"
            for opt in valid_options:
                if opt.lower() in ans.lower():
                    matched_label = opt
                    break
                    
            if matched_label != "Uncategorized":
                matched_cid = [cid for cid, sim in top_5 if symptom_cluster_labels[cid]['label'] == matched_label][0]
                llm_count += 1
        except Exception:
            pass

    if matched_cid != -1:
        symptom_labels[idx] = matched_cid

    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(noise_indices)} noise records...")

print(f"\nSuccessfully recovered {auto_count + llm_count} noise records (Auto: {auto_count}, LLM: {llm_count}).")

print("9. Re-saving enriched dataset...")
df["symptom_cluster_id"] = -1
df["symptom_canonical"] = "Uncategorized"
valid_indices = df.index[valid_mask]
for idx_pos, orig_idx in enumerate(valid_indices):
    cid = int(symptom_labels[idx_pos])
    df.at[orig_idx, "symptom_cluster_id"] = cid
    if cid != -1 and cid in symptom_cluster_labels:
        df.at[orig_idx, "symptom_canonical"] = symptom_cluster_labels[cid]["label"]

df.to_csv(file_path, index=False, encoding="utf-8-sig")
print("Done.")